# Hafta 12: Web Madenciliği ve Tavsiye Sistemleri

Bu notebook'ta aşağıdaki konular ele alınmaktadır:
1. Web madenciliği türleri için pratik örnekler
2. PageRank (manuel + NetworkX)
3. HITS algoritması (manuel + NetworkX)
4. Web usage mining (transaction analizi)
5. Content-based recommendation
6. Collaborative filtering (user-based ve item-based)
7. Matrix factorization (SVD)
8. Öneri sistemi metrikleri (RMSE, MAE, Precision@K, Recall@K, NDCG@K)

## 1. Kütüphaneler

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import networkx as nx

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error, mean_absolute_error, ndcg_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)

print('Kütüphaneler yüklendi.')

## 2. Veri Yükleme

- `transactions.csv`: işlem bazlı ürün sepetleri
- `ecommerce_products.csv`: ürün içerik bilgileri

In [ ]:
BASE = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '..', 'Veri_Setleri_Genel')
transactions_path = os.path.join(BASE, 'transactions.csv')
products_path = os.path.join(BASE, 'ecommerce_products.csv')

df_tx = pd.read_csv(transactions_path)
df_products = pd.read_csv(products_path)

print('transactions:', df_tx.shape)
print('products    :', df_products.shape)
display(df_tx.head())
display(df_products.head())

## 3. Web Structure Mining: PageRank

In [ ]:
# Küçük bir web grafı
edges = [
    ('A', 'B'), ('A', 'C'),
    ('B', 'C'), ('B', 'D'),
    ('C', 'A'), ('C', 'D'), ('C', 'E'),
    ('D', 'E'),
    ('E', 'B'), ('E', 'F'),
    ('F', 'C')
]

G = nx.DiGraph()
G.add_edges_from(edges)

plt.figure(figsize=(6, 5))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=1600, arrows=True, font_weight='bold')
plt.title('Örnek Web Link Ağı')
plt.show()

In [ ]:
def pagerank_manual(graph, d=0.85, iterations=100):
    nodes = list(graph.nodes())
    n = len(nodes)
    idx = {node: i for i, node in enumerate(nodes)}

    # Sütun-stokastik geçiş matrisi
    M = np.zeros((n, n))
    for src in nodes:
        out_links = list(graph.successors(src))
        j = idx[src]
        if len(out_links) == 0:
            M[:, j] = 1.0 / n
        else:
            prob = 1.0 / len(out_links)
            for dst in out_links:
                i = idx[dst]
                M[i, j] = prob

    v = np.ones(n) / n
    teleport = np.ones(n) / n

    for _ in range(iterations):
        v = d * (M @ v) + (1 - d) * teleport

    return {nodes[i]: float(v[i]) for i in range(n)}

pr_manual = pagerank_manual(G, d=0.85, iterations=100)
pr_nx = nx.pagerank(G, alpha=0.85)

pr_df = pd.DataFrame({
    'Node': list(pr_nx.keys()),
    'PageRank_Manual': [pr_manual[k] for k in pr_nx.keys()],
    'PageRank_NetworkX': [pr_nx[k] for k in pr_nx.keys()]
}).sort_values('PageRank_NetworkX', ascending=False)

display(pr_df.round(4))

## 4. Web Structure Mining: HITS

In [ ]:
def hits_manual(graph, max_iter=100):
    nodes = list(graph.nodes())
    h = {n: 1.0 for n in nodes}
    a = {n: 1.0 for n in nodes}

    for _ in range(max_iter):
        # Authority update
        for node in nodes:
            a[node] = sum(h[pred] for pred in graph.predecessors(node))

        # Hub update
        for node in nodes:
            h[node] = sum(a[succ] for succ in graph.successors(node))

        # Normalize
        norm_a = np.sqrt(sum(v * v for v in a.values()))
        norm_h = np.sqrt(sum(v * v for v in h.values()))
        if norm_a > 0:
            a = {k: v / norm_a for k, v in a.items()}
        if norm_h > 0:
            h = {k: v / norm_h for k, v in h.items()}

    return h, a

h_manual, a_manual = hits_manual(G, max_iter=100)
h_nx, a_nx = nx.hits(G, max_iter=200, normalized=True)

hits_df = pd.DataFrame({
    'Node': list(G.nodes()),
    'Hub_Manual': [h_manual[n] for n in G.nodes()],
    'Hub_NetworkX': [h_nx[n] for n in G.nodes()],
    'Authority_Manual': [a_manual[n] for n in G.nodes()],
    'Authority_NetworkX': [a_nx[n] for n in G.nodes()]
}).sort_values('Authority_NetworkX', ascending=False)

display(hits_df.round(4))

## 5. Web Usage Mining: Transaction Davranış Analizi

In [ ]:
# Sepetleri parse et
df_tx['item_list'] = df_tx['Items'].apply(lambda s: [x.strip() for x in str(s).split(',') if x.strip()])

# Item frekansları
all_items = [item for basket in df_tx['item_list'] for item in basket]
item_freq = pd.Series(all_items).value_counts()

display(item_freq.to_frame('frequency'))

plt.figure(figsize=(8, 4))
item_freq.plot(kind='bar', color='steelblue')
plt.title('Ürün Görülme Sıklığı (Web Usage)')
plt.xlabel('Ürün')
plt.ylabel('Frekans')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Recommender: User-Item Matrisinin Kurulması

`TransactionID` kullanıcı, `Items` ürün etkileşimi (implicit feedback = 1) olarak ele alınmıştır.

In [ ]:
records = []
for _, row in df_tx.iterrows():
    user = int(row['TransactionID'])
    for item in row['item_list']:
        records.append((user, item, 1.0))

df_inter = pd.DataFrame(records, columns=['user_id', 'item', 'interaction'])
user_item = df_inter.pivot_table(index='user_id', columns='item', values='interaction', aggfunc='max', fill_value=0.0)

print('User-item matrix boyutu:', user_item.shape)
display(user_item.head())

## 7. Content-Based Filtering

Ürün adı + kategori metni ve sayısal özellikler (`Price`, `Rating`, `Reviews_Count`) birlikte kullanılır.

In [ ]:
products = df_products.copy()
products['content'] = products['Product_Name'].astype(str) + ' ' + products['Category'].astype(str)

tfidf = TfidfVectorizer()
text_vec = tfidf.fit_transform(products['content'])

num_cols = ['Price', 'Rating', 'Reviews_Count']
scaler = MinMaxScaler()
num_scaled = scaler.fit_transform(products[num_cols])

# Metin + sayısal benzerlik (ağırlıklı toplam)
sim_text = cosine_similarity(text_vec)
sim_num = cosine_similarity(num_scaled)
sim_content = 0.7 * sim_text + 0.3 * sim_num

product_index = pd.Series(products.index, index=products['Product_Name']).drop_duplicates()

def recommend_content(product_name, topn=5):
    idx = product_index[product_name]
    sims = list(enumerate(sim_content[idx]))
    sims = sorted(sims, key=lambda x: x[1], reverse=True)[1:topn+1]
    rec_idx = [i for i, _ in sims]
    out = products.loc[rec_idx, ['Product_Name', 'Category', 'Price', 'Rating']].copy()
    out['Similarity'] = [s for _, s in sims]
    return out

sample_product = products['Product_Name'].iloc[0]
print('Seçilen ürün:', sample_product)
display(recommend_content(sample_product, topn=5).round(3))

## 8. Collaborative Filtering

### 8.1 User-Based CF

In [ ]:
user_sim = pd.DataFrame(cosine_similarity(user_item), index=user_item.index, columns=user_item.index)

def recommend_user_based(target_user, topn=5, n_neighbors=5):
    sims = user_sim.loc[target_user].drop(target_user).sort_values(ascending=False).head(n_neighbors)
    neighbors = sims.index

    # Komşu kullanıcıların ortalama etkileşimi
    neighbor_pref = user_item.loc[neighbors].mean(axis=0)

    # Hedef kullanıcının görmediği ürünler
    unseen = user_item.loc[target_user] == 0
    scores = neighbor_pref[unseen].sort_values(ascending=False)
    return scores.head(topn)

target_user = user_item.index[0]
print('Hedef kullanıcı (TransactionID):', target_user)
display(recommend_user_based(target_user, topn=5).to_frame('score'))

### 8.2 Item-Based CF

In [ ]:
item_sim = pd.DataFrame(cosine_similarity(user_item.T), index=user_item.columns, columns=user_item.columns)

def recommend_item_based(target_user, topn=5):
    liked_items = user_item.columns[user_item.loc[target_user] > 0]
    if len(liked_items) == 0:
        return pd.Series(dtype=float)

    # Kullanıcının beğendiği ürünlerin benzerlik skorlarını topla
    sim_scores = item_sim[liked_items].sum(axis=1)

    # Zaten etkileşim verilenleri çıkar
    sim_scores = sim_scores.drop(labels=liked_items, errors='ignore')
    return sim_scores.sort_values(ascending=False).head(topn)

display(recommend_item_based(target_user, topn=5).to_frame('score'))

## 9. Matrix Factorization (SVD)

In [ ]:
R = user_item.values
k = min(3, min(R.shape) - 1)

svd = TruncatedSVD(n_components=k, random_state=42)
U = svd.fit_transform(R)
Vt = svd.components_
R_hat = U @ Vt

pred_matrix = pd.DataFrame(R_hat, index=user_item.index, columns=user_item.columns)

def recommend_svd(target_user, topn=5):
    seen = user_item.loc[target_user] > 0
    scores = pred_matrix.loc[target_user][~seen].sort_values(ascending=False)
    return scores.head(topn)

display(recommend_svd(target_user, topn=5).to_frame('pred_score'))

## 10. Değerlendirme Metrikleri

Bu örnekte implicit veri (0/1) kullanıldığı için basit bir holdout senaryosu ile değerlendirme yapılır.

In [ ]:
np.random.seed(42)
R_full = user_item.copy()
R_train = R_full.copy()

heldout = []  # (user, item)
for user in R_train.index:
    user_items = R_train.columns[R_train.loc[user] > 0]
    if len(user_items) >= 2:
        hide_item = np.random.choice(user_items)
        R_train.loc[user, hide_item] = 0.0
        heldout.append((user, hide_item))

# Train matrisine SVD uygula
R_tr = R_train.values
k_eval = min(3, min(R_tr.shape) - 1)
svd_eval = TruncatedSVD(n_components=k_eval, random_state=42)
Ue = svd_eval.fit_transform(R_tr)
Vte = svd_eval.components_
R_pred = Ue @ Vte
R_pred_df = pd.DataFrame(R_pred, index=R_train.index, columns=R_train.columns)

# RMSE / MAE (heldout etkileşimler için gerçek değer = 1)
y_true = []
y_score = []
for u, i in heldout:
    y_true.append(1.0)
    y_score.append(float(R_pred_df.loc[u, i]))

rmse = np.sqrt(mean_squared_error(y_true, y_score)) if len(y_true) > 0 else np.nan
mae = mean_absolute_error(y_true, y_score) if len(y_true) > 0 else np.nan

print(f'RMSE (heldout-positives): {rmse:.4f}')
print(f'MAE  (heldout-positives): {mae:.4f}')

In [ ]:
def precision_recall_at_k(pred_df, train_df, heldout_pairs, k=3):
    heldout_map = {}
    for u, i in heldout_pairs:
        heldout_map.setdefault(u, set()).add(i)

    precisions = []
    recalls = []
    ndcgs = []

    for u, true_items in heldout_map.items():
        seen = train_df.loc[u] > 0
        scores = pred_df.loc[u][~seen].sort_values(ascending=False)
        rec_items = list(scores.head(k).index)

        hit_count = len(set(rec_items) & true_items)
        precisions.append(hit_count / k)
        recalls.append(hit_count / len(true_items))

        # NDCG@K
        y_true_vec = [1 if item in true_items else 0 for item in rec_items]
        y_score_vec = [float(scores[item]) for item in rec_items]
        if sum(y_true_vec) == 0:
            ndcgs.append(0.0)
        else:
            ndcgs.append(float(ndcg_score([y_true_vec], [y_score_vec], k=k)))

    return np.mean(precisions), np.mean(recalls), np.mean(ndcgs)

p_at_k, r_at_k, ndcg_at_k = precision_recall_at_k(R_pred_df, R_train, heldout, k=3)

print(f'Precision@3: {p_at_k:.4f}')
print(f'Recall@3   : {r_at_k:.4f}')
print(f'NDCG@3     : {ndcg_at_k:.4f}')

## 11. Sonuç

Bu uygulamada hem web madenciliği hem de tavsiye sistemlerinin temel yapı taşları birlikte gösterildi:
- **Web Structure Mining**: PageRank ve HITS ile düğüm önem analizi
- **Web Usage Mining**: işlem verisinden davranış/frekans çıkarımı
- **Recommendation**: content-based, collaborative filtering ve SVD
- **Evaluation**: RMSE, MAE, Precision@K, Recall@K, NDCG@K

Bir sonraki adım olarak gerçek kullanıcı-ürün puanlama verisi ile (ör. MovieLens) aynı pipeline daha güçlü şekilde genişletilebilir.